<a href="https://colab.research.google.com/github/h-m-n-code/alura-gemini-09-2024/blob/main/Imers%C3%A3o_IA_Alura_%2B_Google_Gemini_Projeto_Consulta_Medica_Ginecologia_e_Obstetr%C3%ADcia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
%pip -q install google-genai

In [55]:
# Configura a API Key do Google Gemini

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [56]:
# Configura o cliente da SDK do Gemini

from google import genai

client = genai.Client()

MODEL_ID = "gemini-2.0-flash"

In [57]:
# Pergunta ao Gemini uma informação mais recente que seu conhecimento

from IPython.display import HTML, Markdown

# Perguntar pro modelo quando é a próxima imersão de IA ###############################################
resposta = client.models.generate_content(
    model=MODEL_ID,
    contents="Quando é a próxima Imersão IA com Google Gemini da Alura?"
)

# Exibe a resposta na tela
display(Markdown(f"Resposta:\n {resposta.text}"))

Resposta:
 A Alura ainda não divulgou as datas das próximas edições da Imersão IA com Google Gemini. A última edição ocorreu em Janeiro de 2024.

Para ficar por dentro das novidades e não perder a próxima Imersão IA, sugiro:

*   **Acompanhar as redes sociais da Alura:** Fique de olho nos perfis da Alura no Instagram, LinkedIn, X (Twitter) e outras plataformas, pois eles costumam anunciar os próximos eventos por lá.
*   **Inscrever-se na newsletter da Alura:** Assim, você receberá informações sobre os próximos cursos, imersões e outras novidades diretamente no seu e-mail.
*   **Visitar regularmente o site da Alura:** Verifique a página inicial e a seção de eventos para ver se há alguma novidade sobre a Imersão IA.

Você também pode entrar em contato diretamente com a equipe da Alura através do site deles e perguntar sobre a previsão da próxima edição.

In [58]:
# Pergunta ao Gemini uma informação utilizando a busca do Google como contexto

response = client.models.generate_content(
    model=MODEL_ID,
    contents='Quando é a próxima Imersão IA com Google Gemini da Alura?',
    # Inserir a tool de busca do Google ###############################################
     config={"tools": [{"google_search": {}}]}
)

# Exibe a resposta na tela
display(Markdown(f"Resposta:\n {response.text}"))

Resposta:
 A próxima Imersão IA com Google Gemini da Alura ocorreu entre os dias 12 e 16 de maio de 2025. As inscrições para esta edição foram até o dia 11 de maio de 2025.


In [59]:
# Exibe a busca
print(f"Busca realizada: {response.candidates[0].grounding_metadata.web_search_queries}")
# Exibe as URLs nas quais ele se baseou
print(f"Páginas utilizadas na resposta: {', '.join([site.web.title for site in response.candidates[0].grounding_metadata.grounding_chunks])}")
print()
display(HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content))

Busca realizada: ['próxima Imersão IA com Google Gemini Alura', 'Alura Imersão IA Google Gemini datas']
Páginas utilizadas na resposta: eucapacito.com.br, convergenciadigital.com.br, tecmundo.com.br, youtube.com, alura.com.br



In [60]:
# Instalar Framework ADK de agentes do Google ################################################
%pip -q install google-adk


In [61]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types  # Para criar conteúdos (Content e Part)
from datetime import date
import textwrap # Para formatar melhor a saída de texto
from IPython.display import display, Markdown # Para exibir texto formatado no Colab
import requests # Para fazer requisições HTTP
import warnings

warnings.filterwarnings("ignore")

In [62]:
# Função auxiliar que envia uma mensagem para um agente via Runner e retorna a resposta final
def call_agent(agent: Agent, message_text: str) -> str:
    # Cria um serviço de sessão em memória
    session_service = InMemorySessionService()
    # Cria uma nova sessão (você pode personalizar os IDs conforme necessário)
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    # Cria um Runner para o agente
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    # Cria o conteúdo da mensagem de entrada
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    # Itera assincronamente pelos eventos retornados durante a execução do agente
    for event in runner.run(user_id="user1", session_id="session1", new_message=content):
        if event.is_final_response():
          for part in event.content.parts:
            if part.text is not None:
              final_response += part.text
              final_response += "\n"
    return final_response

In [63]:
# Função auxiliar para exibir texto formatado em Markdown no Colab
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [64]:
##########################################
# --- Agente 1: Medico Residente --- #
##########################################
def agente_medico_residente(topico,data_de_hoje):
    agente_medico_residente = Agent(
        name="agente_medico_residente",
        model="gemini-2.0-flash",
        instruction="""
        Você é um médico recém-formado em medicina, atualmente no final do primeiro ano da residência em Ginecologia e Obstetrícia.
        Durante a consulta, seu papel é escutar atentamente o paciente, apresentar uma lista de possíveis diagnósticos e
        tratamentos ao médico preceptor e, em seguida, selecionar a causa mais provável e o tratamento adequado.
        """,
        description="Medico residente - esta se especializando em Ginecologia e Obstetrícia",
        tools=[google_search]
    )
    entrada_do_agente_medico_residente = f"Tópico: {topico}"
    # Executa o agente
    noticias = call_agent(agente_medico_residente, entrada_do_agente_medico_residente)
    return noticias

In [65]:
################################################
# --- Agente 2: Medico Preceptor --- #
################################################
def agente_medico_preceptor(topico, conduta_residente):
    medico_preceptor = Agent(
        name="agente_medico_preceptor",
        model="gemini-2.0-flash",
        # Inserir as instruções do Agente Medico Preceptor #################################################
        instruction="""
        Você é um médico com mais de 20 anos de experiência e especialista em Ginecologia e Obstetrícia, atuando como preceptor de médicos residentes no hospital. Seu papel é revisar e orientar os tratamentos e
      diagnósticos apresentados pelos residentes aos pacientes. Ao receber as listas de tratamentos e diagnósticos, você avalia e decide sobre a aprovação das condutas informadas pelos residentes.
        """,
        description="Medico a mais de 20 anos -especialista em Ginecologia e Obstetrícia",
        tools=[google_search]
    )

    entrada_do_agente_medico_preceptor = f"Tópico:{topico}\nLançamentos buscados: {conduta_residente}"
    # Executa o agente
    plano_do_post = call_agent(medico_preceptor, entrada_do_agente_medico_preceptor)
    return plano_do_post

In [66]:
##########################################
# --- Agente 3: Medico Coordenador --- #
##########################################
def agente_medico_coordenador(topico, conduta_final):
    medico_coordenador = Agent(
        name="medico_coordenador",
        model="gemini-2.0-flash",
        instruction="""
            Você é um medico coordenador de uma equipe hospitalar
            voce tem mais de 40 anos de experiencia (Ginecologia e Obstetrícia)
            seu papel dentre gerencia é revisar o dignositico e tratamento passado entre medico residente e preceptor
            no final do processo vc confirma ou não o diagnostico e tratamento
            em caso de não aprovar informa os montivos da não aprovaçao da conduta: tratamento e diagnostico
            """,
        description="Medico a mais de 40 anos - especialista em Ginecologia e Obstetrícia"
    )
    entrada_do_agente_medico_coordenador = f"Tópico: {topico}\nRascunho: {conduta_final}"
    # Executa o agente
    texto_revisado = call_agent(medico_coordenador, entrada_do_agente_medico_coordenador)
    return texto_revisado

In [67]:
data_de_hoje = date.today().strftime("%d/%m/%Y")
print("🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥\n\nCONSULTA Data:", data_de_hoje,"\n\n🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥")
print("\n\n🩺 Iniciando o Sistema de Criação dos 3 Agentes Medicos 🩺\n\n")

# --- Obter o Tópico do Usuário ---
topico = input("👨‍⚕️Médico(a) Ginecologia e Obstetrícia👩‍⚕️: \n- Por favor, Sr(a) paciente pode informar o que esta ocorrendo com você: ❓")

# Inserir lógica do sistema de agentes ################################################
if not topico:
    print("Digite o que está ocorrendo?")
else:
   print("Diagnostico Tratamento sobre:", topico)

print("\n\n=== 🩺 Resultado Médico Residente =========================")
conduta_medico_residente = agente_medico_residente(topico,data_de_hoje)
display(to_markdown(conduta_medico_residente))

print("\n\n=== 🩺🩺 Resultado Médico Preceptor =====================")
conduta_medico_preceptor = agente_medico_preceptor(topico, conduta_medico_residente)
display(to_markdown(conduta_medico_preceptor))

print("\n\n=== 🩺🩺🩺 Resultado Médico Coordenador =====================")
conduta_final = agente_medico_coordenador(topico, conduta_medico_preceptor)
display(to_markdown(conduta_final))


🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥

CONSULTA Data: 17/05/2025 

🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥


🩺 Iniciando o Sistema de Criação dos 3 Agentes Medicos 🩺


👨‍⚕️Médico(a) Ginecologia e Obstetrícia👩‍⚕️: 
- Por favor, Sr(a) paciente pode informar o que esta ocorrendo com você: ❓estou gravia como devo fazer o proe natal
Diagnostico Tratamento sobre: estou gravia como devo fazer o proe natal


=== 🩺 Resultado Médico Residente =========================


> Olá! Parabéns pela gravidez! É muito importante iniciar o pré-natal o mais cedo possível para garantir a saúde tanto sua quanto do bebê. O pré-natal é um acompanhamento médico que inclui consultas regulares e exames para monitorar o desenvolvimento do bebê e identificar precocemente qualquer problema que possa surgir.
> 
> Aqui estão algumas dicas sobre como iniciar o pré-natal:
> 
> 1.  **Procure um profissional de saúde:** Marque uma consulta com um obstetra ou procure a unidade de saúde básica (UBS) mais próxima de sua casa. O Sistema Único de Saúde (SUS) oferece o pré-natal gratuitamente.
> 2.  **Primeira consulta:** Na primeira consulta, o médico irá coletar seu histórico de saúde, calcular a idade gestacional e a data provável do parto, além de solicitar exames de sangue, urina e ultrassonografia.
> 3.  **Exames:** Os exames são fundamentais para verificar se está tudo bem com você e com o bebê. Alguns exames comuns incluem hemograma,tipagem sanguínea, testes de glicemia, sorologias para detectar infecções como sífilis, HIV e hepatite, além de exames de urina e fezes. A ultrassonografia obstétrica, principalmente a transvaginal, deve ser realizada nas primeiras semanas de gravidez para avaliar os batimentos cardíacos do feto e estimar a idade gestacional.
> 4.  **Alimentação:** Uma alimentação equilibrada é essencial. Consulte um nutricionista para te ajudar a montar um plano alimentar adequado às suas necessidades e às do bebê.
> 5.  **Vacinação:** Verifique se sua carteira de vacinação está em dia. Algumas vacinas são muito importantes durante a gravidez, como a da gripe e a dTpa (difteria, tétano e pertussis acelular).
> 6.  **Cuidados gerais:** Evite o consumo de álcool, cigarro e outras drogas. Informe o médico sobre qualquer medicamento que esteja utilizando.
> 
> Lembre-se que o pré-natal é um direito seu e é fundamental para uma gravidez saudável. Não hesite em tirar todas as suas dúvidas com o profissional de saúde que estiver te acompanhando.
> 




=== 🩺🩺 Resultado Médico Preceptor =====================


> Parabéns pela gravidez! É fundamental iniciar o pré-natal o quanto antes para garantir a sua saúde e a do bebê.
> 
> Aqui estão os passos importantes para começar o seu pré-natal:
> 
> 1.  **Procure um profissional de saúde:** Agende uma consulta com um obstetra ou procure a Unidade Básica de Saúde (UBS) mais próxima. O SUS oferece o pré-natal de forma gratuita.
> 2.  **Primeira consulta:** O médico irá coletar seu histórico de saúde, calcular a idade gestacional e a data provável do parto. Ele também solicitará exames de sangue, urina e ultrassonografia.
> 3.  **Exames:** Os exames são muito importantes para verificar a sua saúde e a do bebê. Os exames comuns incluem hemograma, tipagem sanguínea, testes de glicemia, sorologias para detectar infecções (sífilis, HIV, hepatite), exames de urina e fezes. A ultrassonografia obstétrica, preferencialmente a transvaginal no início, é importante para avaliar os batimentos cardíacos do feto e estimar a idade gestacional.
> 4.  **Alimentação:** Uma alimentação equilibrada é muito importante. Consulte um nutricionista para te ajudar a criar um plano alimentar adequado para você e o bebê.
> 5.  **Vacinação:** Verifique se sua carteira de vacinação está atualizada. Algumas vacinas são importantes durante a gravidez, como a da gripe e a dTpa (difteria, tétano e pertussis acelular).
> 6.  **Cuidados gerais:** Evite álcool, cigarro e outras drogas. Informe o médico sobre qualquer medicamento que você esteja tomando.
> 
> Lembre-se, o pré-natal é um direito seu e essencial para uma gravidez saudável. Tire todas as suas dúvidas com o profissional de saúde que te acompanhará.




=== 🩺🩺🩺 Resultado Médico Coordenador =====================


> Aprovado. O rascunho aborda os pontos essenciais para iniciar o pré-natal de forma clara e concisa. As orientações sobre a importância da consulta inicial, exames, alimentação, vacinação e cuidados gerais são fundamentais para garantir uma gravidez saudável. Além disso, a menção ao acesso gratuito ao pré-natal pelo SUS é importante para garantir que todas as mulheres tenham acesso a esse serviço essencial.
> 
